In [ ]:
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn_evaluation import plot

In [ ]:
MAX_DEPTH = -30.0

# Get all the csvs in the data directory
data_dir = Path("data")
regions = gpd.read_file('postcards.geojson')

# Read them and merge them into a single dataframe
all = []
atolls = []
islands = []

for region in regions.itertuples():
    csv = data_dir / f"training/{region.name}_land_mask.csv"
    is_atoll = region.type == "Atoll"

    gdf = gpd.read_file(csv)

    # Make sure everything can be converted to a float
    for col in gdf.columns:
        gdf[col] = gdf[col].astype(float)

    # Replace infinite values with NaN
    gdf = gdf.replace([float('-inf'), float('inf')], float('nan'))

    # Drop rows with missing values
    gdf = gdf.dropna()
    gdf = gdf[gdf.depth > MAX_DEPTH]

    print(f"Read {csv} ({'Atoll' if is_atoll else 'Island'}) with {len(gdf)} data points less than {MAX_DEPTH} m")

    # Get them all and put them in lists
    all.append(gdf)
    if is_atoll:
        atolls.append(gdf)
    else:
        islands.append(gdf)

all_data = gpd.GeoDataFrame(pd.concat(all, ignore_index=True))
atolls_data = gpd.GeoDataFrame(pd.concat(atolls, ignore_index=True))
islands_data = gpd.GeoDataFrame(pd.concat(islands, ignore_index=True))

print(f"\nTotal data points: {len(all_data)}, Atolls: {len(atolls_data)}, Islands: {len(islands_data)}")

# Get rid of specifc values, which I think come from inaccurate digitisation of contours
filtered = all_data[~all_data.depth.isin([0, -2, -5, -8, -10, -12.5, -15, -17.5, -20, -25, -30])]

print(f"Filtered data points excluting round numbers: {len(filtered)}")

In [ ]:
# # Run the training using a combination of adaboost, random forest and gradient boosting
# # along with all data, atolls and islands only and store the results in a dictionary, nested by data type and model

# results = {}

# for data, name in [(all_data, 'All'), (atolls_data, 'Atolls'), (islands_data, 'Islands')]:
#     print(f"\n{name} data")
#     X = data.drop(columns=['x', 'y', 'depth'])
#     y = data['depth']

#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#     for model in [AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor]:
#         model_name = model.__name__
#         print(f"- {model_name}")
#         reg = model()
#         reg.fit(X_train, y_train)

#         y_pred = reg.predict(X_test)
#         mse = mean_squared_error(y_test, y_pred)
#         mae = mean_absolute_error(y_test, y_pred)

#         print(f"  - MSE: {mse}")
#         print(f"  - MAE: {mae}")

#         if name not in results:
#             results[name] = {}
#         results[name][model_name] = {
#             'model': reg,
#             'mse': mse,
#             'mae': mae
#         }

In [ ]:
# # Print the results as a markdown table
# print("## Results")
# print("\n| Data | Model | MSE | MAE |")
# print("|------|-------|-----|-----|")
# for data, models in results.items():
#     for model, metrics in models.items():
#         print(f"| {data} | {model} | {metrics['mse']:0.3f} | {metrics['mae']:0.3f} |")


In [ ]:
# Split the data into training and testing
train_first, test_first = train_test_split(filtered, test_size=0.3)

# Define the variables and the target
depth_first = train_first["depth"]
variables_first = train_first.drop(columns=["depth", "x", "y"])

# Define the model
regressor = RandomForestRegressor()

# Train the model
rf_model = regressor.fit(variables_first, depth_first)

# Evaluate on our test data
test_depth_first = test_first["depth"]
test_variables_first = test_first.drop(columns=["depth", "x", "y"])


predictions_first = rf_model.predict(test_variables_first)

# Evaluate the model
r2_first = r2_score(test_depth_first, predictions_first)
mse_first = mean_squared_error(test_depth_first, predictions_first)
mae_first = mean_absolute_error(test_depth_first, predictions_first)

print(f"R² score: {r2_first:.3f}")
print(f"Mean squared error: {mse_first:.3f}")
print(f"Mean absolute error: {mae_first:.3f}")

In [ ]:
# Clean up training data
rf_preds = rf_model.predict(filtered.drop(columns=["depth", "x", "y"]))
residuals = np.abs(rf_preds - filtered["depth"])

threshold = 3.0
mask = residuals < threshold

filtered_data = filtered[mask]

print(f"Filtered data points: {len(filtered_data)} out of {len(all_data)}")

In [ ]:
# # Compare models

# from sklearn.ensemble import (
#     AdaBoostRegressor,
#     RandomForestRegressor,
#     GradientBoostingRegressor,
# )

# X = filtered_data.drop(columns=["x", "y", "depth"])
# y = filtered_data["depth"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# results = {}

# for model in [AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor]:
#     model_name = model.__name__
#     print(f"- {model_name}")
#     reg = model()
#     reg.fit(X_train, y_train)

#     y_pred = reg.predict(X_test)
#     r2 = r2_score(y_test, y_pred)
#     mse = mean_squared_error(y_test, y_pred)
#     mae = mean_absolute_error(y_test, y_pred)

#     print(f"  - R² score: {r2:.3f}")
#     print(f"  - MSE: {mse}")
#     print(f"  - MAE: {mae}")

#     results[model_name] = {"model": reg, "mse": mse, "mae": mae}

In [ ]:
# Split the data into training and testing
train, test = train_test_split(filtered_data, test_size=0.3)

# Define the variables and the target
depth = train["depth"]
variables = train.drop(columns=["depth", "x", "y"])

# Define the model
regressor = RandomForestRegressor()

# Train the model
rf_model = regressor.fit(variables, depth)

# Evaluate on our test data
test_depth = test["depth"]
test_variables = test.drop(columns=["depth", "x", "y"])


predictions = rf_model.predict(test_variables)
mse = mean_squared_error(test_depth, predictions)
mae = mean_absolute_error(test_depth, predictions)
r2 = r2_score(test_depth, predictions)

print(f"R² score: {r2:.3f}")
print(f"Mean squared error: {mse:.3f}")
print(f"Mean absolute error: {mae:.3f}")

In [ ]:
full_depth = filtered_data["depth"]

full_predictions = regressor.predict(filtered_data.drop(columns=["depth", "x", "y"]))
full_residuals = np.abs(full_predictions - full_depth)

filtered_data["residuals"] = full_residuals

In [ ]:
# Plot count of residuals in 5 m bins of depth, from -30 to 0
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.hist(filtered_data["depth"], bins=np.arange(-30, 1, 5), weights=full_depth["residuals"], edgecolor='black')
plt.xlabel("Depth (m)")
plt.ylabel("Count of Residuals")
plt.title("Count of Residuals in 5 m Bins of Depth")
plt.grid()
plt.show()

In [ ]:
depth_bin_mean_residual = []
# Plot mean residuals in 5 m bins of depth, from -30 to 0
for bin in np.arange(-30, 1, 5):
    mask = (filtered_data["depth"] >= bin) & (filtered_data["depth"] < bin + 5)
    mean_residual = filtered_data[mask]["residuals"].mean()
    depth_bin_mean_residual.append((f"{bin} to {bin + 5} m", mean_residual))
    print(f"Mean residual for depth {bin} to {bin + 5} m: {mean_residual:.3f}")

In [ ]:
# Plot depth_bin_mean_residual
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.bar(
    [x[0] for x in depth_bin_mean_residual],
    [x[1] for x in depth_bin_mean_residual],
    color="#00B0FF",
    edgecolor="#00B0FF"
)
plt.xlabel("Depth (m)")
plt.ylabel("Mean Residual")
plt.title("Mean Residuals in 5 m Bins of Depth")
plt.xticks(rotation=45)
# plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(test_depth, predictions, alpha=0.05)
plt.xlabel("True Depth")
plt.ylabel("Predicted Depth")
plt.title("Training Predictions vs True Depth")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from matplotlib import pyplot as plt
import numpy as np

from sklearn_evaluation.plot.regression import _set_ax_settings

# Do this plot, but with alpha on the points
# plot.residuals(test_depth, predictions)

y_true = test_depth
y_pred = predictions


_, ax = plt.subplots()

default_color = "#00B0FF"

# horizontal line for residual=0
ax.axhline(y=0, color=default_color)
ax.scatter(y_pred, y_true - y_pred, c=default_color, edgecolors=default_color, alpha=0.01)

_set_ax_settings(ax, "Predicted Value", "Residuals", "Residuals Plot")

In [ ]:
plot.feature_importances(rf_model, feature_names=variables.columns)

In [ ]:
# Do this plot, but with alpha on the points
# plot.regression.prediction_error(test_depth, predictions)

_, ax = plt.subplots()
regression = LinearRegression()

if isinstance(y_true, pd.Series):
    y_true = y_true.values
y_reshaped = y_true.reshape((-1, 1))

# it is necessary to fit the model with y_true and y_pred
# to get the best fit line representing the error trend
regression.fit(y_reshaped, y_pred)
x = np.linspace(min(min(y_true), min(y_pred)), max(max(y_true), max(y_pred)))
y = regression.intercept_ + regression.coef_ * x

default_color = "#00B0FF"

ax.plot(x, y, color="#666", label="best fit", linewidth=1)

# identity line
ax.plot(
    x, x, label="identity", color="#000", linewidth=1, alpha=0.5, linestyle="dashed"
)

# scatter plot
ax.scatter(y_true, y_pred, c=default_color, edgecolors=default_color, alpha=0.01)

# R2
r2 = regression.score(y_reshaped, y_pred)
plt.plot([], [], " ", label=f"R2 = {round(r2, 5)}")

_set_ax_settings(ax, "Test depth", "Predicted depth", "Prediction Error")
ax.legend(loc="upper left")

In [ ]:
out = "models/2025_06_13_rf.joblib"

# Write out the model
joblib.dump(rf_model, out)

# Write out a little metadata file
with open(out.replace(".joblib", ".txt"), "w") as f:
    f.write(f"R² score: {r2:.3f}\n")
    f.write(f"Mean squared error: {mse:.3f}\n")
    f.write(f"Mean absolute error: {mae:.3f}\n")
    f.write(f"Model: {regressor.__class__.__name__}\n")
    f.write(f"Data: all, limit at {MAX_DEPTH} m\n")
    f.write("Masked: land only\n")
    f.write(f"Date and time: {datetime.now()}\n")
    f.write(f"File: {out}\n")

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

# Zip it up!
with ZipFile(out.replace(".joblib", ".zip"), "w", compression=ZIP_DEFLATED) as z:
    z.write(out)
    z.write(out.replace(".joblib", ".txt"))